In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
import os
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.model_selection import train_test_split,cross_val_score, KFold ,GridSearchCV
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
import numpy as np


In [3]:
#Load the dataset
DATA_DIR = '../data/raw/'

data = pd.read_csv(os.path.join(DATA_DIR, 'sales.csv'), parse_dates=['Date'])

data

,Date,Revenue,COGS
0,2012-07-04,5123547.94,3982991.19
1,2012-07-05,2751773.45,2150580.23
2,2012-07-06,3054029.42,2517632.84
3,2012-07-07,2667930.94,2108246.62
4,2012-07-08,2360851.90,1808622.79
...,...,...,...
3828,2022-12-27,2100553.66,2184872.24
3829,2022-12-28,3448729.20,3513621.00
3830,2022-12-29,3083944.33,3170787.10
3831,2022-12-30,2884668.76,3022292.15


In [4]:
data['year'] = data['Date'].dt.year
data['month'] = data['Date'].dt.month
data['day'] = data['Date'].dt.day
data['dayofweek'] = data['Date'].dt.dayofweek
data['dayofyear'] = data['Date'].dt.dayofyear
data['weekofyear'] = data['Date'].dt.isocalendar().week
data['is_weekend'] = data['dayofweek'].isin([5, 6]).astype(int)
holidays_and_sales = ['01-01', '02-14', '03-08', '04-30', '05-01', '09-02', '10-20', '11-11', '12-12', '12-24']
data['is_holiday'] = data['Date'].dt.strftime('%m-%d').isin(holidays_and_sales).astype(int)
data['sin_1'] = np.sin(2 * np.pi * data['dayofyear'] / 365.25)
data['cos_1'] = np.cos(2 * np.pi * data['dayofyear'] / 365.25)
data['sin_2'] = np.sin(2 * np.pi * data['dayofyear'] / (365.25/2))
data['cos_2'] = np.cos(2 * np.pi * data['dayofyear'] / (365.25/2))
data

,Date,Revenue,COGS,year,month,day,dayofweek,dayofyear,weekofyear,is_weekend,is_holiday,sin_1,cos_1,sin_2,cos_2
0,2012-07-04,5123547.94,3982991.19,2012,7,4,2,186,27,0,0,-0.058026,-0.998315,0.115856,0.993266
1,2012-07-05,2751773.45,2150580.23,2012,7,5,3,187,27,0,0,-0.075190,-0.997169,0.149953,0.988693
2,2012-07-06,3054029.42,2517632.84,2012,7,6,4,188,27,0,0,-0.092331,-0.995728,0.183874,0.982950
3,2012-07-07,2667930.94,2108246.62,2012,7,7,5,189,27,1,0,-0.109446,-0.993993,0.217577,0.976043
4,2012-07-08,2360851.90,1808622.79,2012,7,8,6,190,27,1,0,-0.126528,-0.991963,0.251022,0.967981
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3828,2022-12-27,2100553.66,2184872.24,2022,12,27,1,361,52,0,0,-0.073045,0.997329,-0.145700,0.989329
3829,2022-12-28,3448729.20,3513621.00,2022,12,28,2,362,52,0,0,-0.055879,0.998438,-0.111583,0.993755
3830,2022-12-29,3083944.33,3170787.10,2022,12,29,3,363,52,0,0,-0.038696,0.999251,-0.077334,0.997005
3831,2022-12-30,2884668.76,3022292.15,2022,12,30,4,364,52,0,0,-0.021501,0.999769,-0.042993,0.999075


In [5]:
lags = [1, 7, 14, 28]
windows = [7, 14, 28]

for lag in lags:
    data[f'Revenue_lag_{lag}'] = data['Revenue'].shift(lag)
    data[f'COGS_lag_{lag}'] = data['COGS'].shift(lag)

for w in windows:
    data[f'Revenue_roll_mean_{w}'] = data['Revenue'].shift(1).rolling(w).mean()
    data[f'Revenue_roll_std_{w}'] = data['Revenue'].shift(1).rolling(w).std()
    data[f'COGS_roll_mean_{w}'] = data['COGS'].shift(1).rolling(w).mean()
    data[f'COGS_roll_std_{w}'] = data['COGS'].shift(1).rolling(w).std()
data['Gross_Margin_lag_1'] = (data['Revenue'].shift(1) - data['COGS'].shift(1)) / (data['Revenue'].shift(1) + 1)
data = data.dropna().reset_index(drop=True)
data

,Date,Revenue,COGS,year,month,day,dayofweek,dayofyear,weekofyear,is_weekend,...,COGS_roll_std_7,Revenue_roll_mean_14,Revenue_roll_std_14,COGS_roll_mean_14,COGS_roll_std_14,Revenue_roll_mean_28,Revenue_roll_std_28,COGS_roll_mean_28,COGS_roll_std_28,Gross_Margin_lag_1
0,2012-08-01,9148357.30,7308514.33,2012,8,1,2,214,31,0,...,1.254643e+06,5.234863e+06,1.499371e+06,4.126740e+06,1.153004e+06,4.657385e+06,1.441663e+06,3.661303e+06,1.116172e+06,0.219495
1,2012-08-02,9692427.00,7537501.15,2012,8,2,3,215,31,0,...,1.584917e+06,5.518021e+06,1.827479e+06,4.357919e+06,1.431915e+06,4.801128e+06,1.672098e+06,3.780071e+06,1.311508e+06,0.201112
2,2012-08-03,9297269.95,7118808.46,2012,8,3,4,216,31,0,...,1.812585e+06,5.751660e+06,2.135074e+06,4.538650e+06,1.661457e+06,5.049008e+06,1.860845e+06,3.972461e+06,1.451284e+06,0.222331
3,2012-08-04,6384195.03,5087528.11,2012,8,4,5,217,31,1,...,1.767519e+06,6.046360e+06,2.325112e+06,4.746137e+06,1.793896e+06,5.271981e+06,1.982981e+06,4.136789e+06,1.538336e+06,0.234312
4,2012-08-05,2359224.19,1863413.11,2012,8,5,6,218,31,1,...,1.452543e+06,6.256505e+06,2.201291e+06,4.913383e+06,1.699763e+06,5.404705e+06,1.925775e+06,4.243192e+06,1.495262e+06,0.203106
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3800,2022-12-27,2100553.66,2184872.24,2022,12,27,1,361,52,0,...,3.592945e+05,1.699922e+06,3.454986e+05,1.727411e+06,3.540577e+05,1.515372e+06,4.890883e+05,1.548556e+06,5.104657e+05,-0.023751
3801,2022-12-28,3448729.20,3513621.00,2022,12,28,2,362,52,0,...,3.838777e+05,1.731940e+06,3.611613e+05,1.762933e+06,3.741305e+05,1.499009e+06,4.596675e+05,1.530359e+06,4.760260e+05,-0.040141
3802,2022-12-29,3083944.33,3170787.10,2022,12,29,3,363,52,0,...,7.856147e+05,1.859546e+06,5.824504e+05,1.893320e+06,5.974926e+05,1.575117e+06,5.872499e+05,1.607761e+06,6.039972e+05,-0.018816
3803,2022-12-30,2884668.76,3022292.15,2022,12,30,4,364,52,0,...,9.005524e+05,1.973090e+06,6.560662e+05,2.008776e+06,6.777455e+05,1.625671e+06,6.528450e+05,1.660893e+06,6.724271e+05,-0.028160


In [6]:
data['mean_rev_by_month'] = data.groupby('month')['Revenue'].transform('mean')
data['mean_cogs_by_month'] = data.groupby('month')['COGS'].transform('mean')
data['mean_rev_by_week'] = data.groupby('weekofyear')['Revenue'].transform('mean')
data['mean_cogs_by_week'] = data.groupby('weekofyear')['COGS'].transform('mean')
data['mean_rev_by_dayofweek'] = data.groupby('dayofweek')['Revenue'].transform('mean')
data['mean_cogs_by_dayofweek'] = data.groupby('dayofweek')['COGS'].transform('mean')
data

,Date,Revenue,COGS,year,month,day,dayofweek,dayofyear,weekofyear,is_weekend,...,Revenue_roll_std_28,COGS_roll_mean_28,COGS_roll_std_28,Gross_Margin_lag_1,mean_rev_by_month,mean_cogs_by_month,mean_rev_by_week,mean_cogs_by_week,mean_rev_by_dayofweek,mean_cogs_by_dayofweek
0,2012-08-01,9148357.30,7308514.33,2012,8,1,2,214,31,0,...,1.441663e+06,3.661303e+06,1.116172e+06,0.219495,4.441193e+06,4.348378e+06,6.118653e+06,5.676566e+06,4.675622e+06,4.027144e+06
1,2012-08-02,9692427.00,7537501.15,2012,8,2,3,215,31,0,...,1.672098e+06,3.780071e+06,1.311508e+06,0.201112,4.441193e+06,4.348378e+06,6.118653e+06,5.676566e+06,4.518140e+06,3.904244e+06
2,2012-08-03,9297269.95,7118808.46,2012,8,3,4,216,31,0,...,1.860845e+06,3.972461e+06,1.451284e+06,0.222331,4.441193e+06,4.348378e+06,6.118653e+06,5.676566e+06,4.041783e+06,3.493369e+06
3,2012-08-04,6384195.03,5087528.11,2012,8,4,5,217,31,1,...,1.982981e+06,4.136789e+06,1.538336e+06,0.234312,4.441193e+06,4.348378e+06,6.118653e+06,5.676566e+06,3.909096e+06,3.375928e+06
4,2012-08-05,2359224.19,1863413.11,2012,8,5,6,218,31,1,...,1.925775e+06,4.243192e+06,1.495262e+06,0.203106,4.441193e+06,4.348378e+06,6.118653e+06,5.676566e+06,4.077655e+06,3.516713e+06
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3800,2022-12-27,2100553.66,2184872.24,2022,12,27,1,361,52,0,...,4.890883e+05,1.548556e+06,5.104657e+05,-0.023751,2.524350e+06,2.439383e+06,3.524860e+06,3.389916e+06,4.455712e+06,3.835238e+06
3801,2022-12-28,3448729.20,3513621.00,2022,12,28,2,362,52,0,...,4.596675e+05,1.530359e+06,4.760260e+05,-0.040141,2.524350e+06,2.439383e+06,3.524860e+06,3.389916e+06,4.675622e+06,4.027144e+06
3802,2022-12-29,3083944.33,3170787.10,2022,12,29,3,363,52,0,...,5.872499e+05,1.607761e+06,6.039972e+05,-0.018816,2.524350e+06,2.439383e+06,3.524860e+06,3.389916e+06,4.518140e+06,3.904244e+06
3803,2022-12-30,2884668.76,3022292.15,2022,12,30,4,364,52,0,...,6.528450e+05,1.660893e+06,6.724271e+05,-0.028160,2.524350e+06,2.439383e+06,3.524860e+06,3.389916e+06,4.041783e+06,3.493369e+06


In [13]:
current_features = [
    'year', 'month', 'day', 'is_weekend', 'is_holiday',
    'dayofweek', 'dayofyear', 'weekofyear', 'sin_1', 'cos_1', 'sin_2', 'cos_2',
    'mean_rev_by_week', 'mean_rev_by_month', 'mean_rev_by_dayofweek',
    'mean_cogs_by_week', 'mean_cogs_by_month', 'mean_cogs_by_dayofweek'
]
lag_features_cogs = [f'COGS_lag_{lag}' for lag in lags]
lag_features_revenue = [f'Revenue_lag_{lag}' for lag in lags]
roll_features_cogs = [f'COGS_roll_mean_{w}' for w in windows] + [f'COGS_roll_std_{w}' for w in windows]
roll_features_revenue = [f'Revenue_roll_mean_{w}' for w in windows] + [f'Revenue_roll_std_{w}' for w in windows]
features_for_cogs = current_features + lag_features_cogs + roll_features_cogs
features_for_revenue = current_features + lag_features_revenue + lag_features_cogs + roll_features_revenue + roll_features_cogs + ['Gross_Margin_lag_1']
print(f"Features COGS: {len(features_for_cogs)}")
print(f"Features Revenue: {len(features_for_revenue)}")

Features COGS: 28
Features Revenue: 39


In [14]:
train_mask = data['Date'] < '2022-01-01'
val_mask = (data['Date'] >= '2022-01-01') & (data['Date'] <= '2022-12-31')

X_train_revenue = data.loc[train_mask, features_for_revenue]
y_train_revenue = data.loc[train_mask, 'Revenue']
X_val_revenue = data.loc[val_mask, features_for_revenue]
y_val_revenue = data.loc[val_mask, 'Revenue']

X_train_cogs = data.loc[train_mask, features_for_cogs]
y_train_cogs = data.loc[train_mask, 'COGS']
X_val_cogs = data.loc[val_mask, features_for_cogs]
y_val_cogs = data.loc[val_mask, 'COGS']

In [15]:
model_lgb_rev = LGBMRegressor(n_estimators=2000, learning_rate=0.02, num_leaves=64,subsample=0.8, colsample_bytree=0.8, random_state=42, verbose=-1)
model_xgb_rev = xgb.XGBRegressor(n_estimators=2000, learning_rate=0.02, max_depth=6,subsample=0.8, colsample_bytree=0.8, random_state=42, tree_method="hist", verbosity=0)
model_cat_rev = CatBoostRegressor(loss_function='RMSE', iterations=1000, learning_rate=0.1, depth=6, random_seed=42, verbose=0)

model_lgb_rev.fit(X_train_revenue, y_train_revenue)
model_xgb_rev.fit(X_train_revenue, y_train_revenue)
model_cat_rev.fit(X_train_revenue, y_train_revenue)

oof_lgb_rev = model_lgb_rev.predict(X_val_revenue)
oof_xgb_rev = model_xgb_rev.predict(X_val_revenue)
oof_cat_rev = model_cat_rev.predict(X_val_revenue)

oof_blend_rev = 0.4 * oof_lgb_rev + 0.35 * oof_cat_rev + 0.25 * oof_xgb_rev

print(f"LightGBM - MAE: {mean_absolute_error(y_val_revenue, oof_lgb_rev):,.0f}, RMSE: {np.sqrt(mean_squared_error(y_val_revenue, oof_lgb_rev)):,.0f}, R²: {r2_score(y_val_revenue, oof_lgb_rev):.4f}")
print(f"XGBoost - MAE: {mean_absolute_error(y_val_revenue, oof_xgb_rev):,.0f}, RMSE: {np.sqrt(mean_squared_error(y_val_revenue, oof_xgb_rev)):,.0f}, R²: {r2_score(y_val_revenue, oof_xgb_rev):.4f}")
print(f"CatBoost - MAE: {mean_absolute_error(y_val_revenue, oof_cat_rev):,.0f}, RMSE: {np.sqrt(mean_squared_error(y_val_revenue, oof_cat_rev)):,.0f}, R²: {r2_score(y_val_revenue, oof_cat_rev):.4f}")
print(f"Blend - MAE: {mean_absolute_error(y_val_revenue, oof_blend_rev):,.0f}, RMSE: {np.sqrt(mean_squared_error(y_val_revenue, oof_blend_rev)):,.0f}, R²: {r2_score(y_val_revenue, oof_blend_rev):.4f}")

LightGBM - MAE: 539,189, RMSE: 727,543, R²: 0.8111
XGBoost - MAE: 532,157, RMSE: 721,748, R²: 0.8141
CatBoost - MAE: 521,079, RMSE: 700,801, R²: 0.8247
Blend - MAE: 522,420, RMSE: 702,942, R²: 0.8236


In [16]:
stack_train_rev = np.vstack([oof_lgb_rev, oof_cat_rev, oof_xgb_rev]).T
lvl2_rev = Ridge(alpha=1.0)
lvl2_rev.fit(stack_train_rev, y_val_revenue)
pred_stack_rev = lvl2_rev.predict(stack_train_rev)

print(f"Stacked Ridge - MAE: {mean_absolute_error(y_val_revenue, pred_stack_rev):,.0f}, RMSE: {np.sqrt(mean_squared_error(y_val_revenue, pred_stack_rev)):,.0f}, R²: {r2_score(y_val_revenue, pred_stack_rev):.6f}")
print(f"Ridge coefs: {lvl2_rev.coef_}")

Stacked Ridge - MAE: 514,742, RMSE: 687,978, R²: 0.831059
Ridge coefs: [0.09301013 0.76464865 0.07284695]


In [17]:
model_lgb_cogs = LGBMRegressor(n_estimators=2000, learning_rate=0.02, num_leaves=64,subsample=0.8, colsample_bytree=0.8, random_state=42, verbose=-1)
model_xgb_cogs = xgb.XGBRegressor(n_estimators=2000, learning_rate=0.02, max_depth=6,subsample=0.8, colsample_bytree=0.8, random_state=42, tree_method="hist", verbosity=0)
model_cat_cogs = CatBoostRegressor(loss_function='RMSE', iterations=1000, learning_rate=0.1,depth=6, random_seed=42, verbose=0)

model_lgb_cogs.fit(X_train_cogs, y_train_cogs)
model_xgb_cogs.fit(X_train_cogs, y_train_cogs)
model_cat_cogs.fit(X_train_cogs, y_train_cogs)

oof_lgb_cogs = model_lgb_cogs.predict(X_val_cogs)
oof_xgb_cogs = model_xgb_cogs.predict(X_val_cogs)
oof_cat_cogs = model_cat_cogs.predict(X_val_cogs)

oof_blend_cogs = 0.4 * oof_lgb_cogs + 0.35 * oof_cat_cogs + 0.25 * oof_xgb_cogs

print(f"LightGBM - MAE: {mean_absolute_error(y_val_cogs, oof_lgb_cogs):,.0f}, RMSE: {np.sqrt(mean_squared_error(y_val_cogs, oof_lgb_cogs)):,.0f}, R²: {r2_score(y_val_cogs, oof_lgb_cogs):.4f}")
print(f"XGBoost  - MAE: {mean_absolute_error(y_val_cogs, oof_xgb_cogs):,.0f}, RMSE: {np.sqrt(mean_squared_error(y_val_cogs, oof_xgb_cogs)):,.0f}, R²: {r2_score(y_val_cogs, oof_xgb_cogs):.4f}")
print(f"CatBoost - MAE: {mean_absolute_error(y_val_cogs, oof_cat_cogs):,.0f}, RMSE: {np.sqrt(mean_squared_error(y_val_cogs, oof_cat_cogs)):,.0f}, R²: {r2_score(y_val_cogs, oof_cat_cogs):.4f}")
print(f"Blend    - MAE: {mean_absolute_error(y_val_cogs, oof_blend_cogs):,.0f}, RMSE: {np.sqrt(mean_squared_error(y_val_cogs, oof_blend_cogs)):,.0f}, R²: {r2_score(y_val_cogs, oof_blend_cogs):.4f}")

stack_train_cogs = np.vstack([oof_lgb_cogs, oof_cat_cogs, oof_xgb_cogs]).T
lvl2_cogs = Lasso(alpha=0.1, max_iter=5000)
lvl2_cogs.fit(stack_train_cogs, y_val_cogs)
pred_stack_cogs = lvl2_cogs.predict(stack_train_cogs)

print(f"Stacked Lasso - MAE: {mean_absolute_error(y_val_cogs, pred_stack_cogs):,.0f}, RMSE: {np.sqrt(mean_squared_error(y_val_cogs, pred_stack_cogs)):,.0f}, R²: {r2_score(y_val_cogs, pred_stack_cogs):.4f}")
print(f"Lasso coefs: {lvl2_cogs.coef_}")

LightGBM - MAE: 479,607, RMSE: 628,063, R²: 0.8146
XGBoost  - MAE: 466,488, RMSE: 620,134, R²: 0.8192
CatBoost - MAE: 461,620, RMSE: 613,634, R²: 0.8230
Blend    - MAE: 458,198, RMSE: 607,414, R²: 0.8266
Stacked Lasso - MAE: 457,903, RMSE: 598,294, R²: 0.8317
Lasso coefs: [0.17832104 0.63433626 0.12396258]


In [18]:
from prophet import Prophet

prophet_train_cogs = data.loc[train_mask, ['Date', 'COGS']].rename(columns={'Date': 'ds', 'COGS': 'y'})
model_prophet_cogs = Prophet(yearly_seasonality=True, weekly_seasonality=True, daily_seasonality=False)
model_prophet_cogs.fit(prophet_train_cogs)

prophet_train_rev = data.loc[train_mask, ['Date', 'Revenue']].rename(columns={'Date': 'ds', 'Revenue': 'y'})
model_prophet_rev = Prophet(yearly_seasonality=True, weekly_seasonality=True, daily_seasonality=False)
model_prophet_rev.fit(prophet_train_rev)

11:49:20 - cmdstanpy - INFO - Chain [1] start processing
11:49:21 - cmdstanpy - INFO - Chain [1] done processing
11:49:22 - cmdstanpy - INFO - Chain [1] start processing
11:49:22 - cmdstanpy - INFO - Chain [1] done processing


In [19]:
prophet_pred_rev_train = model_prophet_rev.predict(pd.DataFrame({'ds': data.loc[train_mask, 'Date']}))['yhat'].values

residuals_rev_train = data.loc[train_mask, 'Revenue'].values - prophet_pred_rev_train

model_lgb_rev.fit(X_train_revenue, residuals_rev_train)
model_xgb_rev.fit(X_train_revenue, residuals_rev_train)
model_cat_rev.fit(X_train_revenue, residuals_rev_train)

oof_lgb_rev_res = model_lgb_rev.predict(X_val_revenue)
oof_xgb_rev_res = model_xgb_rev.predict(X_val_revenue)
oof_cat_rev_res = model_cat_rev.predict(X_val_revenue)

stack_train_rev_res = np.vstack([oof_lgb_rev_res, oof_cat_rev_res, oof_xgb_rev_res]).T
y_val_rev_res = data.loc[val_mask, 'Revenue'].values - model_prophet_rev.predict(pd.DataFrame({'ds': data.loc[val_mask, 'Date']}))['yhat'].values

lvl2_rev = Lasso(alpha=0.1, max_iter=5000)
lvl2_rev.fit(stack_train_rev_res, y_val_rev_res)
print(f"Revenue Lasso coefs: {lvl2_rev.coef_}")

pred_stack_rev_res = lvl2_rev.predict(stack_train_rev_res)
pred_rev_final = model_prophet_rev.predict(pd.DataFrame({'ds': data.loc[val_mask, 'Date']}))['yhat'].values + pred_stack_rev_res
print(f"Revenue Final (Prophet + Stacked Residuals) - R²: {r2_score(y_val_revenue, pred_rev_final):.4f}")

Revenue Lasso coefs: [-0.20213939  0.89423268  0.30881787]
Revenue Final (Prophet + Stacked Residuals) - R²: 0.8111


In [20]:
prophet_pred_cogs_train = model_prophet_cogs.predict(pd.DataFrame({'ds': data.loc[train_mask, 'Date']}))['yhat'].values

residuals_cogs_train = data.loc[train_mask, 'COGS'].values - prophet_pred_cogs_train

model_lgb_cogs.fit(X_train_cogs, residuals_cogs_train)
model_xgb_cogs.fit(X_train_cogs, residuals_cogs_train)
model_cat_cogs.fit(X_train_cogs, residuals_cogs_train)

oof_lgb_cogs_res = model_lgb_cogs.predict(X_val_cogs)
oof_xgb_cogs_res = model_xgb_cogs.predict(X_val_cogs)
oof_cat_cogs_res = model_cat_cogs.predict(X_val_cogs)

stack_train_cogs_res = np.vstack([oof_lgb_cogs_res, oof_cat_cogs_res, oof_xgb_cogs_res]).T
y_val_cogs_res = data.loc[val_mask, 'COGS'].values - model_prophet_cogs.predict(pd.DataFrame({'ds': data.loc[val_mask, 'Date']}))['yhat'].values

lvl2_cogs = Lasso(alpha=0.1, max_iter=5000)
lvl2_cogs.fit(stack_train_cogs_res, y_val_cogs_res)
print(f"COGS Lasso coefs: {lvl2_cogs.coef_}")

pred_stack_cogs_res = lvl2_cogs.predict(stack_train_cogs_res)
pred_cogs_final = model_prophet_cogs.predict(pd.DataFrame({'ds': data.loc[val_mask, 'Date']}))['yhat'].values + pred_stack_cogs_res
print(f"COGS Final (Prophet + Stacked Residuals) - R²: {r2_score(y_val_cogs, pred_cogs_final):.4f}")

COGS Lasso coefs: [0.03976154 0.56109484 0.38273736]
COGS Final (Prophet + Stacked Residuals) - R²: 0.8030


In [23]:
def recursive_forecast_with_prophet(historical_data, forecast_dates,
                                    model_prophet_cogs, model_prophet_rev,
                                    model_lgb_c, model_xgb_c, model_cat_c, lvl2_c,
                                    model_lgb_r, model_xgb_r, model_cat_r, lvl2_r,
                                    features_for_cogs, features_for_revenue,
                                    lags, windows, holidays_and_sales):
    historical = historical_data.copy()
    predictions = []
    
    for forecast_date in forecast_dates:
        features = {}

        future_df = pd.DataFrame({'ds': [forecast_date]})
        prophet_cogs_pred = model_prophet_cogs.predict(future_df)['yhat'].values[0]
        prophet_rev_pred = model_prophet_rev.predict(future_df)['yhat'].values[0]

        
        features['year'] = forecast_date.year
        features['month'] = forecast_date.month
        features['day'] = forecast_date.day
        features['dayofweek'] = forecast_date.dayofweek
        features['is_weekend'] = 1 if forecast_date.dayofweek in [5, 6] else 0
        features['dayofyear'] = forecast_date.dayofyear
        features['weekofyear'] = forecast_date.isocalendar().week
        features['is_holiday'] = 1 if forecast_date.strftime('%m-%d') in holidays_and_sales else 0
        for k in [1, 2]:
            features[f'sin_{k}'] = np.sin(2 * np.pi * k * features['dayofyear'] / 365.25)
            features[f'cos_{k}'] = np.cos(2 * np.pi * k * features['dayofyear'] / 365.25)

        features['mean_rev_by_month'] = historical.groupby('month')['Revenue'].mean().get(features['month'], historical['Revenue'].mean())
        features['mean_cogs_by_month'] = historical.groupby('month')['COGS'].mean().get(features['month'], historical['COGS'].mean())
        features['mean_rev_by_week'] = historical.groupby('weekofyear')['Revenue'].mean().get(features['weekofyear'], historical['Revenue'].mean())
        features['mean_cogs_by_week'] = historical.groupby('weekofyear')['COGS'].mean().get(features['weekofyear'], historical['COGS'].mean())
        features['mean_rev_by_dayofweek'] = historical.groupby('dayofweek')['Revenue'].mean().get(features['dayofweek'], historical['Revenue'].mean())
        features['mean_cogs_by_dayofweek'] = historical.groupby('dayofweek')['COGS'].mean().get(features['dayofweek'], historical['COGS'].mean())
        

        for lag in lags:
            idx = len(historical) - lag
            features[f'COGS_lag_{lag}'] = historical['COGS'].iloc[idx] if idx >= 0 else historical['COGS'].mean()
        
        for w in windows:
            if len(historical) >= w:
                features[f'COGS_roll_mean_{w}'] = historical['COGS'].iloc[-w:].mean()
                features[f'COGS_roll_std_{w}'] = historical['COGS'].iloc[-w:].std()
            else:
                features[f'COGS_roll_mean_{w}'] = historical['COGS'].mean()
                features[f'COGS_roll_std_{w}'] = 0
        
        X_cogs = pd.DataFrame([features]).reindex(columns=features_for_cogs, fill_value=0)
        
        p_lgb_c = model_lgb_c.predict(X_cogs)[0]
        p_xgb_c = model_xgb_c.predict(X_cogs)[0]
        p_cat_c = model_cat_c.predict(X_cogs)[0]
        cogs_residual = lvl2_c.predict(np.array([[p_lgb_c, p_cat_c, p_xgb_c]]))[0]

        pred_cogs = prophet_cogs_pred + cogs_residual
        pred_cogs = max(0, pred_cogs)

        for lag in lags:
            idx = len(historical) - lag
            features[f'Revenue_lag_{lag}'] = historical['Revenue'].iloc[idx] if idx >= 0 else historical['Revenue'].mean()
        
        for w in windows:
            if len(historical) >= w:
                features[f'Revenue_roll_mean_{w}'] = historical['Revenue'].iloc[-w:].mean()
                features[f'Revenue_roll_std_{w}'] = historical['Revenue'].iloc[-w:].std()
            else:
                features[f'Revenue_roll_mean_{w}'] = historical['Revenue'].mean()
                features[f'Revenue_roll_std_{w}'] = 0
        
        if len(historical) >= 1:
            rev_l1 = historical['Revenue'].iloc[-1]
            cogs_l1 = historical['COGS'].iloc[-1]
            features['Gross_Margin_lag_1'] = (rev_l1 - cogs_l1) / (rev_l1 + 1)
        else:
            features['Gross_Margin_lag_1'] = 0
        
        X_rev = pd.DataFrame([features]).reindex(columns=features_for_revenue, fill_value=0)
        
        p_lgb_r = model_lgb_r.predict(X_rev)[0]
        p_xgb_r = model_xgb_r.predict(X_rev)[0]
        p_cat_r = model_cat_r.predict(X_rev)[0]
        rev_residual = lvl2_r.predict(np.array([[p_lgb_r, p_cat_r, p_xgb_r]]))[0]

        pred_rev = prophet_rev_pred + rev_residual
        pred_rev = max(0, pred_rev)

        predictions.append({
            'Date': forecast_date,
            'COGS': pred_cogs,
            'Revenue': pred_rev
        })

        new_row = historical.iloc[-1].copy()
        new_row['Date'] = forecast_date
        new_row['COGS'] = pred_cogs
        new_row['Revenue'] = pred_rev
        historical = pd.concat([historical, pd.DataFrame([new_row])], ignore_index=True)
    
    return pd.DataFrame(predictions)

In [24]:
val_dates = pd.date_range('2022-01-01', '2022-12-31', freq='D')

recursive_val = recursive_forecast_with_prophet(
    data[data['Date'] < '2022-01-01'],
    val_dates,
    model_prophet_cogs, model_prophet_rev,
    model_lgb_cogs, model_xgb_cogs, model_cat_cogs, lvl2_cogs,
    model_lgb_rev, model_xgb_rev, model_cat_rev, lvl2_rev,
    features_for_cogs, features_for_revenue,
    lags, windows, holidays_and_sales
)

comparison = recursive_val.merge(
    data.loc[val_mask, ['Date', 'Revenue', 'COGS']], 
    on='Date', 
    suffixes=('_pred', '_actual')
)

print(f"Revenue:")
print(f"  RMSE: {np.sqrt(mean_squared_error(comparison['Revenue_actual'], comparison['Revenue_pred'])):,.0f}")
print(f"  MAE:  {mean_absolute_error(comparison['Revenue_actual'], comparison['Revenue_pred']):,.0f}")
print(f"  R²:   {r2_score(comparison['Revenue_actual'], comparison['Revenue_pred']):.4f}")

print(f"\nCOGS:")
print(f"  RMSE: {np.sqrt(mean_squared_error(comparison['COGS_actual'], comparison['COGS_pred'])):,.0f}")
print(f"  MAE:  {mean_absolute_error(comparison['COGS_actual'], comparison['COGS_pred']):,.0f}")
print(f"  R²:   {r2_score(comparison['COGS_actual'], comparison['COGS_pred']):.4f}")

Revenue:
  RMSE: 869,389
  MAE:  616,433
  R²:   0.7302

COGS:
  RMSE: 701,316
  MAE:  524,799
  R²:   0.7688


In [25]:
model_lgb_cogs_final = LGBMRegressor(n_estimators=2000, learning_rate=0.02, num_leaves=64, subsample=0.8, colsample_bytree=0.8, random_state=42, verbose=-1)
model_xgb_cogs_final = xgb.XGBRegressor(n_estimators=2000, learning_rate=0.02, max_depth=6, subsample=0.8, colsample_bytree=0.8, random_state=42, tree_method="hist", verbosity=0)
model_cat_cogs_final = CatBoostRegressor(loss_function='RMSE', iterations=1000, learning_rate=0.1, depth=6, random_seed=42, verbose=0)

model_lgb_rev_final = LGBMRegressor(n_estimators=2000, learning_rate=0.02, num_leaves=64, subsample=0.8, colsample_bytree=0.8, random_state=42, verbose=-1)
model_xgb_rev_final = xgb.XGBRegressor(n_estimators=2000, learning_rate=0.02, max_depth=6, subsample=0.8, colsample_bytree=0.8, random_state=42, tree_method="hist", verbosity=0)
model_cat_rev_final = CatBoostRegressor(loss_function='RMSE', iterations=1000, learning_rate=0.1, depth=6, random_seed=42, verbose=0)

In [26]:
model_prophet_cogs_final = Prophet(yearly_seasonality=True, weekly_seasonality=True, daily_seasonality=False)
model_prophet_cogs_final.fit(data[['Date', 'COGS']].rename(columns={'Date': 'ds', 'COGS': 'y'}))

model_prophet_rev_final = Prophet(yearly_seasonality=True, weekly_seasonality=True, daily_seasonality=False)
model_prophet_rev_final.fit(data[['Date', 'Revenue']].rename(columns={'Date': 'ds', 'Revenue': 'y'}))

prophet_cogs_all = model_prophet_cogs_final.predict(pd.DataFrame({'ds': data['Date']}))['yhat'].values
prophet_rev_all = model_prophet_rev_final.predict(pd.DataFrame({'ds': data['Date']}))['yhat'].values

residuals_cogs_all = data['COGS'].values - prophet_cogs_all
residuals_rev_all = data['Revenue'].values - prophet_rev_all

model_lgb_cogs_final.fit(data[features_for_cogs], residuals_cogs_all)
model_xgb_cogs_final.fit(data[features_for_cogs], residuals_cogs_all)
model_cat_cogs_final.fit(data[features_for_cogs], residuals_cogs_all)

oof_cogs_res_lgb = model_lgb_cogs_final.predict(data[features_for_cogs])
oof_cogs_res_xgb = model_xgb_cogs_final.predict(data[features_for_cogs])
oof_cogs_res_cat = model_cat_cogs_final.predict(data[features_for_cogs])
stack_cogs_full = np.vstack([oof_cogs_res_lgb, oof_cogs_res_cat, oof_cogs_res_xgb]).T
lvl2_cogs_final = Lasso(alpha=0.1, max_iter=10000)
lvl2_cogs_final.fit(stack_cogs_full, residuals_cogs_all)

model_lgb_rev_final.fit(data[features_for_revenue], residuals_rev_all)
model_xgb_rev_final.fit(data[features_for_revenue], residuals_rev_all)
model_cat_rev_final.fit(data[features_for_revenue], residuals_rev_all)

oof_rev_res_lgb = model_lgb_rev_final.predict(data[features_for_revenue])
oof_rev_res_xgb = model_xgb_rev_final.predict(data[features_for_revenue])
oof_rev_res_cat = model_cat_rev_final.predict(data[features_for_revenue])
stack_rev_full = np.vstack([oof_rev_res_lgb, oof_rev_res_cat, oof_rev_res_xgb]).T
lvl2_rev_final = Lasso(alpha=0.1, max_iter=10000)
lvl2_rev_final.fit(stack_rev_full, residuals_rev_all)

11:58:11 - cmdstanpy - INFO - Chain [1] start processing
11:58:11 - cmdstanpy - INFO - Chain [1] done processing
11:58:12 - cmdstanpy - INFO - Chain [1] start processing
11:58:12 - cmdstanpy - INFO - Chain [1] done processing


,"alpha alpha: float, default=1.0Constant that multiplies the L1 term, controlling regularizationstrength. `alpha` must be a non-negative float i.e. in `[0, inf)`.When `alpha = 0`, the objective is equivalent to ordinary leastsquares, solved by the :class:`LinearRegression` object. For numericalreasons, using `alpha = 0` with the `Lasso` object is not advised.Instead, you should use the :class:`LinearRegression` object.",0.1
,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"precompute precompute: bool or array-like of shape (n_features, n_features), default=FalseWhether to use a precomputed Gram matrix to speed upcalculations. The Gram matrix can also be passed as argument.For sparse input this option is always ``False`` to preserve sparsity.",False
,"copy_X copy_X: bool, default=TrueIf ``True``, X will be copied; else, it may be overwritten.",True
,"max_iter max_iter: int, default=1000The maximum number of iterations.",10000
,"tol tol: float, default=1e-4The tolerance for the optimization: if the updates are smaller or equal to``tol``, the optimization code checks the dual gap for optimality and continuesuntil it is smaller or equal to ``tol``, see Notes below.",0.0001
,"warm_start warm_start: bool, default=FalseWhen set to ``True``, reuse the solution of the previous call to fit asinitialization, otherwise, just erase the previous solution.See :term:`the Glossary `.",False
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive.",False
,"random_state random_state: int, RandomState instance, default=NoneThe seed of the pseudo random number generator that selects a randomfeature to update. Used when ``selection`` == 'random'.Pass an int for reproducible output across multiple function calls.See :term:`Glossary `.",None
,"selection selection: {'cyclic', 'random'}, default='cyclic'If set to 'random', a random coefficient is updated every iterationrather than looping over features sequentially by default. This(setting to 'random') often leads to significantly faster convergenceespecially when tol is higher than 1e-4.",'cyclic'


In [27]:
test_dates = pd.date_range('2023-01-01', '2024-07-01', freq='D')
submission = recursive_forecast_with_prophet(
    data, test_dates,
    model_prophet_cogs_final, model_prophet_rev_final,
    model_lgb_cogs_final, model_xgb_cogs_final, model_cat_cogs_final, lvl2_cogs_final,
    model_lgb_rev_final, model_xgb_rev_final, model_cat_rev_final, lvl2_rev_final,
    features_for_cogs, features_for_revenue,
    lags, windows, holidays_and_sales
)
submission

,Date,COGS,Revenue
0,2023-01-01,2.052711e+06,2.018981e+06
1,2023-01-02,1.646245e+06,1.382106e+06
2,2023-01-03,1.319293e+06,1.296605e+06
3,2023-01-04,8.827207e+05,1.051333e+06
4,2023-01-05,8.948703e+05,1.176032e+06
...,...,...,...
543,2024-06-27,5.314685e+06,4.966653e+06
544,2024-06-28,6.465605e+06,6.049985e+06
545,2024-06-29,6.293591e+06,5.561419e+06
546,2024-06-30,6.357304e+06,5.642975e+06


In [28]:
submission[['Date', 'Revenue', 'COGS']].to_csv(os.path.join('../data/processed/submission_2.csv'), index=False)